In [7]:
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

INPUT_FILE  = "Datasets/Moral Machine Data/SharedResponses.csv"
OUTPUT_FILE = "SharedResponses_clean.parquet"
CHUNK_SIZE  = 500_000

cols_to_drop = [
    "UserID", "ResponseID",
    "LeftHand", "ScenarioType",
    "Template", "DescriptionShown", "ScenarioOrder"
]

char_cols = [
    "Man", "Woman", "Pregnant", "Stroller", "OldMan", "OldWoman",
    "Boy", "Girl", "Homeless", "LargeWoman", "LargeMan", "Criminal",
    "MaleExecutive", "FemaleExecutive", "FemaleAthlete", "MaleAthlete",
    "FemaleDoctor", "MaleDoctor", "Dog", "Cat"
]

bool_cols     = ["Barrier", "Intervention", "DefaultChoiceIsOmission", "Saved", "PedPed"]
category_cols = ["ExtendedSessionID", "UserCountry3", "DefaultChoice", "NonDefaultChoice", "AttributeLevel", "ScenarioTypeStrict"]
uint8_cols    = ["NumberOfCharacters", "CrossingSignal"] + char_cols
int8_cols     = ["DiffNumberOFCharacters"]

def process_chunk(chunk):
    # Step 1 — drop unwanted columns
    chunk = chunk.drop(columns=cols_to_drop, errors="ignore")

    # Step 2 — drop all rows with any nulls
    chunk = chunk.dropna()

    # Step 3 — fix corrupted Man values then cast
    chunk["Man"] = pd.to_numeric(chunk["Man"], errors="coerce")
    chunk = chunk.dropna(subset=["Man"])  # drop the 2 corrupted rows

    # Step 4 — cast dtypes (no nullable types needed since no nulls)
    for col in uint8_cols:
        chunk[col] = chunk[col].astype("uint8")

    for col in int8_cols:
        chunk[col] = chunk[col].astype("int8")

    for col in bool_cols:
        chunk[col] = chunk[col].astype("bool")

    for col in category_cols:
        chunk[col] = chunk[col].astype("category")

    return chunk

# ── Read → process → write ──────────────────────────────────────────────────
writer    = None
total_in  = 0
total_out = 0

for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):
    total_in += len(chunk)
    chunk = process_chunk(chunk)
    total_out += len(chunk)

    table = pa.Table.from_pandas(chunk)
    if writer is None:
        writer = pq.ParquetWriter(OUTPUT_FILE, table.schema)
    writer.write_table(table)

    if i % 10 == 0:
        print(f"Processed {(i + 1) * CHUNK_SIZE:,} rows...")

writer.close()
print(f"\nRows before:  {total_in:,}")
print(f"Rows after:   {total_out:,}")
print(f"Rows dropped: {total_in - total_out:,}")
print(f"Saved to {OUTPUT_FILE}")

Processed 500,000 rows...


/var/folders/l7/7gthwdwn7nb0nr03fdbyh4440000gn/T/ipykernel_89834/2633669176.py:58: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processed 5,500,000 rows...


/var/folders/l7/7gthwdwn7nb0nr03fdbyh4440000gn/T/ipykernel_89834/2633669176.py:58: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processed 10,500,000 rows...
Processed 15,500,000 rows...
Processed 20,500,000 rows...
Processed 25,500,000 rows...
Processed 30,500,000 rows...
Processed 35,500,000 rows...


...

Processed 40,500,000 rows...


/var/folders/l7/7gthwdwn7nb0nr03fdbyh4440000gn/T/ipykernel_89834/2633669176.py:58: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processed 45,500,000 rows...
Processed 50,500,000 rows...
Processed 55,500,000 rows...


/var/folders/l7/7gthwdwn7nb0nr03fdbyh4440000gn/T/ipykernel_89834/2633669176.py:58: DtypeWarning: Columns (0: Man) have mixed types. Specify dtype option on import or set low_memory=False.
  for i, chunk in enumerate(pd.read_csv(INPUT_FILE, chunksize=CHUNK_SIZE)):


Processed 60,500,000 rows...
Processed 65,500,000 rows...
Processed 70,500,000 rows...

Rows before:  70,332,355
Rows after:   62,231,803
Rows dropped: 8,100,552
Saved to SharedResponses_clean.parquet
